# Predict and Verify Test Predictions (Submodule-Local)

This notebook runs prediction using local weights/data/splits in `MixtureDesign`
and compares with local copied reference `predictions.csv` files.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "chemprop/examples/MixtureDesign").exists():
            return p
    raise RuntimeError(f"Could not locate repo root from {start}")


repo_root = find_repo_root(Path.cwd())
mixture_dir = repo_root / "chemprop/examples/MixtureDesign"
runtime_dir = mixture_dir / "runtime"
if str(runtime_dir) not in sys.path:
    sys.path.insert(0, str(runtime_dir))

import train_gnn_ffn as tgm
from utils.mixtures import collate_mixture
from utils.train_common import load_split_dataframes

print(f"mixture_dir: {mixture_dir}")


mixture_dir: /Users/u0161682/Library/CloudStorage/OneDrive-KULeuven/Documents/ScientificOutput/CodeRepositories/GitLab/chempropmix/chemprop/examples/MixtureDesign


In [2]:
jobs = [
    {
        "name": "dcn_mix_exp_fold_00",
        "mix_csv": mixture_dir / "data/dcn_mix_exp.csv",
        "split_csv": mixture_dir / "splits/dcn_mix_exp_mixture_combination_fold_00.csv",
        "model_path": mixture_dir / "weights/dcn_mix_exp_mixture_combination_fold_00_model.pt",
        "reference_predictions_path": mixture_dir / "references/dcn_mix_exp_mixture_combination_fold_00_predictions.csv",
    },
    {
        "name": "flashpoint_mix_exp_fold_00",
        "mix_csv": mixture_dir / "data/flashpoint_mix_exp.csv",
        "split_csv": mixture_dir / "splits/flashpoint_mix_exp_mixture_combination_fold_00.csv",
        "model_path": mixture_dir / "weights/flashpoint_mix_exp_mixture_combination_fold_00_model.pt",
        "reference_predictions_path": mixture_dir / "references/flashpoint_mix_exp_mixture_combination_fold_00_predictions.csv",
    },
    {
        "name": "viscosity_mix_exp_fold_00",
        "mix_csv": mixture_dir / "data/viscosity_mix_exp.csv",
        "split_csv": mixture_dir / "splits/viscosity_mix_exp_mixture_combination_fold_00.csv",
        "model_path": mixture_dir / "weights/viscosity_mix_exp_mixture_combination_fold_00_model.pt",
        "reference_predictions_path": mixture_dir / "references/viscosity_mix_exp_mixture_combination_fold_00_predictions.csv",
    },
]

output_dir = mixture_dir / "repro_checks"
output_dir.mkdir(parents=True, exist_ok=True)
fraction_basis = "mole"
seed = 0


In [3]:
def evaluate_gnn_local(mix_csv: Path, split_csv: Path, model_path: Path):
    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    use_mixmp = ckpt.get("mixmp") is not None

    train_df, val_df, test_df = load_split_dataframes(
        mix_csv=mix_csv,
        split_csv=split_csv,
        max_components=None,
        min_fraction=1e-4,
        fraction_basis=fraction_basis,
        seed=seed,
    )
    target_df = test_df

    train_all_data = tgm.build_all_data(train_df.reset_index(drop=True), target_col="value", solute_component_index=-1)
    n_components = len([c for c in train_df.columns if c.startswith("component_inchi_")])
    train_mcdset = tgm.build_mixture_dataset(train_all_data, n_components, use_mixmp=use_mixmp)
    scaler = train_mcdset.normalize_targets()

    split_all_data = tgm.build_all_data(target_df.reset_index(drop=True), target_col="value", solute_component_index=-1)
    eval_mcdset = tgm.build_mixture_dataset(split_all_data, n_components, use_mixmp=use_mixmp)
    eval_mcdset.normalize_targets(scaler)
    loader = DataLoader(eval_mcdset, batch_size=64, shuffle=False, collate_fn=collate_mixture)

    model = tgm.build_model(
        n_components=n_components,
        scaler=scaler,
        aggregation="weightedsum",
        solute_component_index=-1,
        use_mixmp=use_mixmp,
        x_d_dim=0,
    )
    model.message_passing.load_state_dict(ckpt["message_passing"])
    if model.agg.mixmp is not None and ckpt.get("mixmp") is not None:
        model.agg.mixmp.load_state_dict(ckpt["mixmp"])
    model.agg.load_state_dict(ckpt["mixagg"], strict=False)
    model.predictor.load_state_dict(ckpt["predictor"])
    model.eval()

    preds = []
    with torch.no_grad():
        for batch in loader:
            bmgs, v_ds, x_d_batch, _, _, _, _ = batch
            preds.append(model(bmgs, v_ds, x_d_batch).cpu().numpy().reshape(-1))
    y_pred = np.concatenate(preds, axis=0)
    y_true = target_df["value"].to_numpy(dtype=float).reshape(-1)
    return y_true, y_pred


rows = []
for job in jobs:
    name = job["name"]
    mp = Path(job["model_path"])
    rp = Path(job["reference_predictions_path"])

    if not mp.exists():
        rows.append({"name": name, "status": "missing_model", "model_path": str(mp)})
        print(f"[SKIP] {name}: missing model")
        continue
    if not rp.exists():
        rows.append({"name": name, "status": "missing_reference_predictions", "model_path": str(mp)})
        print(f"[SKIP] {name}: missing reference predictions")
        continue

    y_true, y_pred = evaluate_gnn_local(Path(job["mix_csv"]), Path(job["split_csv"]), mp)
    ref = pd.read_csv(rp)["y_pred"].to_numpy(dtype=float).reshape(-1)
    n = min(len(y_pred), len(ref))
    y_pred, ref, y_true = y_pred[:n], ref[:n], y_true[:n]
    diff = y_pred - ref

    rows.append(
        {
            "name": name,
            "status": "ok",
            "n_compared": int(n),
            "exact_match": bool(np.array_equal(y_pred, ref)),
            "allclose_atol_1e8": bool(np.allclose(y_pred, ref, atol=1e-8, rtol=0.0)),
            "max_abs_diff": float(np.max(np.abs(diff))) if n > 0 else np.nan,
        }
    )

    pd.DataFrame(
        {
            "y_true": y_true,
            "y_pred_recomputed": y_pred,
            "y_pred_reference": ref,
            "delta": diff,
        }
    ).to_csv(output_dir / f"{name}_prediction_comparison.csv", index=False)

    print(f"[OK] {name}: max_abs_diff={float(np.max(np.abs(diff))):.3e}")

summary = pd.DataFrame(rows)
summary.to_csv(output_dir / "mole_gnn_test_repro_summary.csv", index=False)
display(summary)


[OK] dcn_mix_exp_fold_00: max_abs_diff=3.679e-06


[OK] flashpoint_mix_exp_fold_00: max_abs_diff=3.429e-06
[SKIP] viscosity_mix_exp_fold_00: missing model


,name,status,n_compared,exact_match,allclose_atol_1e8,max_abs_diff,model_path
0,dcn_mix_exp_fold_00,ok,46.0,False,False,0.000004,NaN
1,flashpoint_mix_exp_fold_00,ok,89.0,False,False,0.000003,NaN
2,viscosity_mix_exp_fold_00,missing_model,NaN,NaN,NaN,NaN,/Users/u0161682/Library/CloudStorage/OneDrive-...
